# 🏆 DATATHON 2026 — EDA: Stage 4 (Retention & Loyalty)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mticker, seaborn as snsimport warnings; warnings.filterwarnings('ignore')plt.rcParams.update({'figure.figsize':(14,6),'figure.dpi':120,'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','axes.grid':True,'grid.color':'#21262d','grid.alpha':0.6,'text.color':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e','font.size':11,'axes.titlesize':14,'legend.facecolor':'#161b22','legend.edgecolor':'#30363d'})PALETTE = ['#58a6ff','#f78166','#3fb950','#d2a8ff','#f0883e','#79c0ff','#56d364']sns.set_palette(PALETTE)import sys; sys.path.insert(0, '..')from joined.data_pipeline import load_transaction_master, load_reviews_enrichedtxn = load_transaction_master(); rev = load_reviews_enriched()orders = pd.read_csv('../dataset_cleaned/orders.csv', parse_dates=['order_date'])customers = pd.read_csv('../dataset_cleaned/customers.csv', parse_dates=['signup_date'])txn['order_date'] = pd.to_datetime(txn['order_date'])print("✅ Data loaded")

---# 🔄 Giai đoạn 4: Giữ chân & Trung thành (Retention & Loyalty)> **Predictive + Prescriptive:** RFM, CLV, Churn, Price Sensitivity

### 4.1 Phân khúc RFM (Recency–Frequency–Monetary)

In [ ]:
ref_date = txn['order_date'].max() + pd.Timedelta(days=1)delivered = txn[txn['order_status'].isin(['delivered','shipped'])]rfm = delivered.groupby('customer_id').agg(    recency=('order_date', lambda x: (ref_date - x.max()).days),    frequency=('order_id', 'nunique'),    monetary=('line_revenue', 'sum')).reset_index()# Score 1-5for col in ['recency','frequency','monetary']:    reverse = col == 'recency'    rfm[f'{col}_score'] = pd.qcut(rfm[col], 5, labels=[5,4,3,2,1] if reverse else [1,2,3,4,5], duplicates='drop').astype(int)rfm['rfm_score'] = rfm['recency_score']*100 + rfm['frequency_score']*10 + rfm['monetary_score']# Segment mappingdef rfm_segment(row):    r, f, m = row['recency_score'], row['frequency_score'], row['monetary_score']    if r >= 4 and f >= 4 and m >= 4: return 'Champions'    if r >= 3 and f >= 3 and m >= 3: return 'Loyal'    if r >= 4 and f <= 2: return 'New Customers'    if r <= 2 and f >= 3: return 'At Risk'    if r <= 2 and f <= 2: return 'Hibernating'    return 'Potential'rfm['segment'] = rfm.apply(rfm_segment, axis=1)seg_counts = rfm['segment'].value_counts()seg_rev = rfm.groupby('segment')['monetary'].sum().reindex(seg_counts.index)fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))colors = [PALETTE[i%len(PALETTE)] for i in range(len(seg_counts))]ax1.barh(seg_counts.index, seg_counts.values, color=colors)ax1.set_title('Số KH theo Phân khúc RFM', fontweight='bold')for i, v in enumerate(seg_counts.values):    ax1.text(v+200, i, f'{v:,} ({v/len(rfm)*100:.1f}%)', va='center', fontsize=10, color='#c9d1d9')ax2.barh(seg_rev.index, seg_rev.values/1e9, color=colors)ax2.set_title('Doanh thu theo Phân khúc RFM (Tỷ VND)', fontweight='bold')for i, v in enumerate(seg_rev.values/1e9):    ax2.text(v+0.05, i, f'{v:.1f}B', va='center', fontsize=10, color='#c9d1d9')plt.tight_layout(); plt.show()champ = rfm[rfm['segment']=='Champions']print(f"🏆 Champions: {len(champ):,} KH ({len(champ)/len(rfm)*100:.1f}%) đóng góp {champ['monetary'].sum()/rfm['monetary'].sum()*100:.1f}% doanh thu")

### 4.2 RFM Heatmap — Recency vs Frequency

In [ ]:
rf = rfm.groupby(['recency_score','frequency_score'])['monetary'].mean().reset_index()rf_pivot = rf.pivot(index='recency_score', columns='frequency_score', values='monetary')fig, ax = plt.subplots(figsize=(10, 8))sns.heatmap(rf_pivot/1e3, annot=True, fmt=',.0f', cmap='RdYlGn', ax=ax, linewidths=1)ax.set_title('💎 RFM Heatmap: Monetary TB (Nghìn VND)', fontsize=14, fontweight='bold')ax.set_xlabel('Frequency Score →'); ax.set_ylabel('← Recency Score')plt.tight_layout(); plt.show()

### 4.3 Tỷ lệ quay lại & Chu kỳ mua hàng

In [ ]:
cust_orders = orders.sort_values(['customer_id','order_date'])order_counts = cust_orders.groupby('customer_id')['order_id'].nunique()repeat_rate = (order_counts > 1).mean() * 100third_rate = (order_counts > 2).mean() * 100# Inter-order gaprepeat_cust = cust_orders[cust_orders['customer_id'].isin(order_counts[order_counts>1].index)]repeat_cust['prev_date'] = repeat_cust.groupby('customer_id')['order_date'].shift(1)repeat_cust['gap_days'] = (repeat_cust['order_date'] - repeat_cust['prev_date']).dt.daysgaps = repeat_cust['gap_days'].dropna()fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))# Order frequency distributionfreq_dist = order_counts.clip(upper=10).value_counts().sort_index()ax1.bar(freq_dist.index, freq_dist.values, color=PALETTE[0], alpha=0.8)ax1.set_title(f'Phân phối Tần suất mua (Repeat Rate={repeat_rate:.1f}%)', fontweight='bold')ax1.set_xlabel('Số đơn hàng'); ax1.set_ylabel('Số KH')# Gap distributionax2.hist(gaps[gaps<500], bins=50, color=PALETTE[2], alpha=0.7, edgecolor='white')ax2.axvline(gaps.median(), color=PALETTE[1], linestyle='--', linewidth=2, label=f'Median={gaps.median():.0f} ngày')ax2.set_title('Phân phối Inter-order Gap', fontweight='bold')ax2.set_xlabel('Số ngày giữa 2 lần mua'); ax2.legend()plt.tight_layout(); plt.show()print(f"📊 Repeat Rate: {repeat_rate:.1f}% | 3rd Purchase Rate: {third_rate:.1f}%")print(f"📊 Median Inter-order Gap: {gaps.median():.0f} ngày")print(f"   → Thời điểm vàng gửi KM nhắc lại: khoảng ngày thứ {int(gaps.median()*0.8)}")

### 4.4 Cohort Analysis — Tỷ lệ giữ chân theo tháng đăng ký

In [ ]:
orders_c = orders.merge(customers[['customer_id','signup_date']], on='customer_id', how='left')orders_c['signup_cohort'] = orders_c['signup_date'].dt.to_period('Q').astype(str)orders_c['order_quarter'] = orders_c['order_date'].dt.to_period('Q')orders_c['signup_quarter'] = orders_c['signup_date'].dt.to_period('Q')orders_c['cohort_index'] = (orders_c['order_quarter'] - orders_c['signup_quarter']).apply(lambda x: x.n if hasattr(x, 'n') else 0)cohort = orders_c.groupby(['signup_cohort','cohort_index'])['customer_id'].nunique().reset_index()cohort_pivot = cohort.pivot(index='signup_cohort', columns='cohort_index', values='customer_id')# Normalizecohort_pct = cohort_pivot.divide(cohort_pivot.iloc[:, 0], axis=0) * 100# Show last 12 cohorts, first 8 periodsdisplay_data = cohort_pct.iloc[-12:, :8]fig, ax = plt.subplots(figsize=(14, 8))sns.heatmap(display_data, annot=True, fmt='.0f', cmap='YlGnBu', ax=ax, linewidths=0.5,            vmin=0, vmax=100, cbar_kws={'label': 'Retention %'})ax.set_title('📉 Cohort Retention Rate (%) — Last 12 quarters', fontsize=14, fontweight='bold')ax.set_xlabel('Quý sau đăng ký'); ax.set_ylabel('Cohort (Quý đăng ký)')plt.tight_layout(); plt.show()

### 4.5 Độ nhạy cảm giá — Khách hàng 'nghiện' KM vs Mua nguyên giá

In [ ]:
cust_promo = delivered.groupby('customer_id').agg(    total_orders=('order_id', 'nunique'),    promo_orders=('promo_id', lambda x: x.notna().sum()),    total_rev=('line_revenue', 'sum'),    avg_margin=('gross_margin', 'mean')).reset_index()cust_promo['promo_pct'] = cust_promo['promo_orders'] / cust_promo['total_orders'] * 100def price_segment(pct):    if pct == 0: return 'Full Price'    if pct < 50: return 'Occasional Deal'    if pct < 100: return 'Deal Seeker'    return 'Promo Addict'cust_promo['price_seg'] = cust_promo['promo_pct'].apply(price_segment)seg_order = ['Full Price', 'Occasional Deal', 'Deal Seeker', 'Promo Addict']ps_agg = cust_promo.groupby('price_seg').agg(    count=('customer_id','count'), avg_rev=('total_rev','mean'), avg_margin=('avg_margin','mean')).reindex(seg_order)fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))colors = [PALETTE[2], PALETTE[0], PALETTE[3], PALETTE[1]]ax1.bar(ps_agg.index, ps_agg['count'], color=colors)ax1.set_title('Số KH theo Price Sensitivity', fontweight='bold')for i, v in enumerate(ps_agg['count']):    ax1.text(i, v*1.02, f'{v:,}', ha='center', fontsize=10, color='#c9d1d9')ax1.tick_params(axis='x', rotation=15)ax2.bar(ps_agg.index, ps_agg['avg_margin']*100, color=colors)ax2.set_title('Biên LN gộp TB theo nhóm (%)', fontweight='bold')for i, v in enumerate(ps_agg['avg_margin']*100):    ax2.text(i, v+0.3, f'{v:.1f}%', ha='center', fontsize=10, color='#c9d1d9')ax2.tick_params(axis='x', rotation=15)plt.tight_layout(); plt.show()addicts = cust_promo[cust_promo['price_seg']=='Promo Addict']print(f"⚠️ Promo Addicts: {len(addicts):,} KH ({len(addicts)/len(cust_promo)*100:.1f}%) — Margin thấp nhất")print(f"   → Prescriptive: Giảm tần suất gửi mã KM cho nhóm này, tập trung loyalty program")

### 4.6 Churn Analysis — Khách hàng 'ngủ đông'

In [ ]:
last_purchase = orders.groupby('customer_id')['order_date'].max().reset_index()last_purchase.columns = ['customer_id', 'last_order']cutoff = orders['order_date'].max()last_purchase['days_since'] = (cutoff - last_purchase['last_order']).dt.daysdef churn_status(days):    if days <= 90: return 'Active (<3m)'    if days <= 180: return 'Warm (3-6m)'    if days <= 365: return 'Cooling (6-12m)'    return 'Churned (>12m)'last_purchase['status'] = last_purchase['days_since'].apply(churn_status)status_order = ['Active (<3m)', 'Warm (3-6m)', 'Cooling (6-12m)', 'Churned (>12m)']status_counts = last_purchase['status'].value_counts().reindex(status_order)fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))colors = [PALETTE[2], PALETTE[0], PALETTE[3], PALETTE[1]]ax1.pie(status_counts, labels=status_counts.index, autopct='%1.1f%%', colors=colors,        textprops={'color': '#c9d1d9'}, startangle=90)ax1.set_title('Phân bổ trạng thái KH', fontweight='bold')ax2.hist(last_purchase['days_since'], bins=50, color=PALETTE[0], alpha=0.7, edgecolor='white')ax2.axvline(180, color=PALETTE[1], linestyle='--', label='6 tháng')ax2.axvline(365, color='#ff4444', linestyle='--', label='12 tháng')ax2.set_title('Phân phối ngày kể từ lần mua cuối', fontweight='bold')ax2.set_xlabel('Ngày'); ax2.legend()plt.tight_layout(); plt.show()churned = last_purchase[last_purchase['status']=='Churned (>12m)']print(f"🔴 Churned: {len(churned):,} KH ({len(churned)/len(last_purchase)*100:.1f}%)")print(f"   → Prescriptive: Chạy Win-back campaign cho {len(last_purchase[last_purchase['status']=='Cooling (6-12m)']):,} KH đang 'nguội'")

### 4.7 Tiếng nói khách hàng — Rating Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))# Overall ratingrating_dist = rev['rating'].value_counts().sort_index()colors_r = [PALETTE[1] if r<=2 else PALETTE[3] if r==3 else PALETTE[2] for r in rating_dist.index]ax1.bar(rating_dist.index, rating_dist.values, color=colors_r, alpha=0.8)ax1.set_title(f'Phân phối Rating (Mean={rev["rating"].mean():.2f})', fontweight='bold')ax1.set_xlabel('Rating'); ax1.set_ylabel('Số review')# By categorycat_rating = rev.groupby('category')['rating'].mean().sort_values()colors_c = [PALETTE[1] if r<3.5 else PALETTE[2] for r in cat_rating]ax2.barh(cat_rating.index, cat_rating.values, color=colors_c)ax2.axvline(rev['rating'].mean(), color='#8b949e', linestyle='--', alpha=0.5)ax2.set_title('Rating TB theo Danh mục', fontweight='bold')for i, v in enumerate(cat_rating.values):    ax2.text(v+0.02, i, f'{v:.2f}', va='center', fontsize=10, color='#c9d1d9')plt.tight_layout(); plt.show()

---## 📝 Tổng kết EDA — Key Findings| # | Insight | Level | Tác động ||---|---|---|---|| 1 | Doanh thu tăng trưởng YoY nhưng biên LN thu hẹp | Predictive | Cần tối ưu chi phí || 2 | Q4 (T11-12) là mùa vàng, Q1 thấp điểm | Descriptive | Lên kế hoạch inventory || 3 | 59K đơn chưa hoàn tiền = X tỷ VND bị kẹt | Prescriptive | Auto-refund process || 4 | Inventory Health Index chỉ 52/100 | Diagnostic | Overhaul supply chain || 5 | KM đang ăn mòn biên LN, 382 ngày lỗ/năm | Diagnostic | Discount cap policy || 6 | Champions 20% KH đóng góp >50% doanh thu | Descriptive | Loyalty program || 7 | Promo Addicts có margin thấp nhất | Prescriptive | Giảm tần suất KM || 8 | Giao chậm >14 ngày → Return rate & rating xấu | Diagnostic | SLA với vận chuyển |